In [1]:
import numpy as np
import torch

In [2]:
class BatchNorm1d:
    def __init__(
            self, num_features: int, 
            eps: float = 1e-5,
            momentum: float = 0.1,
            train: bool = True
    ):
        self.num_features = num_features
        self.eps = eps
        self.m = momentum
        self.training = train

        self.weight = np.ones((1, self.num_features))
        self.bias = np.zeros((1, self.num_features))

        self.running_mean = np.zeros((1, self.num_features))
        self.running_var = np.ones((1, self.num_features))

        def forward(self, x: np.ndarray) -> np.ndarray:
            if self.training:
                batch_size = x.shape[0]

                self.batch_mean = np.mean(x, axis=0, keepdims=True)
                self.batch_var = np.var(x, axis=0, keepdims=True)

                self.x_norm = (x - self.batch_mean)/(np.sqrt(self.batch_var + self.eps))
                unbiassed_var = self.batch_var * batch_size / (batch_size - 1)


                self.running_mean = ((1 - self.m) * self.running_mean + self.m * self.batch_mean)
                self.running_var = ((1- self.m) * self.running_var + self.m * unbiassed_var)

            else:
                self.x_norm = (x - self.running_mean)/np.sqrt(self.running_var + self.eps)

            output = self.x_norm  * self.weight + self.bias

        def backward(self, grad: np.ndarray) -> np.ndarray:
            std_inv = self.weight / np.sqrt(self.batch_var + self.eps)
            
            grad_weight = (grad * self.x_norm).sum(axis=0, keepdims=True)
            grad_bias = grad.sum(axis=0, keepdims=True)

            grad_x = std_inv * (grad - grad.mean(axis=0, keepdims=True) - self.x_norm * (grad * self.x_norm).mean(axis=0, keepdims=True))

            return grad_x, grad_weight, grad_bias

        def __call__(self, x: np.ndarray) -> np.ndarray:
            return self.forward(x)


In [3]:
input = torch.normal(16, 10, size=(8, 10), dtype=torch.float32, requires_grad=True)
bn = torch.nn.BatchNorm1d(10)
norm = bn(input)

arr = torch.normal(3, 4, size=norm.shape, dtype=torch.float32)

l = norm * arr
loss = l.sum()
loss.backward()

In [5]:
input_np = input.detach().numpy()
arr_np = arr.numpy()

bs, N = input_np.shape
weight = np.ones((1, N), dtype=np.float32)
bias = np.zeros((1, N), dtype=np.float32)

bn_0 = input_np.sum(axis=0, keepdims=True)
bn_mean = bn_0 / bs
bn_1 = input_np - bn_mean
bn_2= bn_1 ** 2
bn_3 = bn_2.sum(axis=0, keepdims=True)
bn_var = bn_3 / bs
bn_sqrt_var = (bn_var + 1e-5) ** 0.5
norm_x = bn_1/ bn_sqrt_var
bn_4 = norm_x * weight
y = bn_4 + bias

l_np = y * arr_np

d_l = np.ones_like(l_np, dtype=np.float32)
dy = d_l * arr_np
dbn_4 = dy
dbias = dy.sum(axis=0, keepdims=True)
dweight = (dbn_4 * norm_x).sum(axis=0, keepdims=True)
dnorm_x = dbn_4 * weight
d_bn_1 = dnorm_x * 1/bn_sqrt_var
d_bn_sqrt_var = -(dnorm_x * bn_1 / (bn_sqrt_var**2)).sum(axis=0, keepdims=True)
d_bn_var = d_bn_sqrt_var * 1/2 * (bn_var + 1e-5) ** -0.5
d_bn_3 = d_bn_var / bs
d_bn_2 = d_bn_3 * np.ones_like(bn_2)
d_bn_1  += d_bn_2 * 2 * bn_1
d_input_np = d_bn_1
d_bn_mean = -d_bn_1.sum(axis=0, keepdims=True)
d_bn_0 = d_bn_mean / bs
d_input_np += d_bn_0 * np.ones_like(bn_2)